# audio_controls

> Audio playback controls: speed selector and auto-navigate toggle

In [ ]:
#| default_exp components.audio_controls

In [ ]:
#| export
from typing import Any

from fasthtml.common import Div, Span, Input, Label

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_input.toggle import toggle, toggle_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, bg_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import m
from cjm_fasthtml_tailwind.utilities.typography import font_size
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

# Web Audio library — shared speed selector component + config
from cjm_fasthtml_web_audio.models import WebAudioConfig
from cjm_fasthtml_web_audio.components import render_speed_selector

## HTML IDs

HTML ID constants for audio control elements.

In [ ]:
#| export
class AudioControlIds:
    """HTML ID constants for audio control elements."""

    AUTO_NAV_TOGGLE = "sd-review-auto-nav-toggle"
    AUDIO_CONTROLS = "sd-review-audio-controls"

## Speed Selector Config

The speed selector itself is provided by `cjm_fasthtml_web_audio.components.render_speed_selector`. We only need a minimal `WebAudioConfig` here (namespace + `enable_speed=True`) to drive its namespace-derived IDs and `window.setReviewSpeed` / `window.cycleReviewSpeed*` wiring.

Defined locally (rather than reusing `REVIEW_AUDIO_CONFIG` from `components.callbacks`) to avoid a circular import — `callbacks` imports symbols from this module.

In [ ]:
#| export
# Minimal config used only to drive the shared render_speed_selector.
# Must stay namespace-compatible with REVIEW_AUDIO_CONFIG (defined in components.callbacks)
# so both configs resolve to the same `window.setReviewSpeed`, `sd-review-speed-select`, etc.
_REVIEW_SPEED_CONFIG = WebAudioConfig(
    namespace="review",
    indicator_selector="",
    enable_speed=True,
)

## Auto-Navigate Toggle

Toggle switch to enable automatic navigation to next segment when audio playback completes.

In [ ]:
#| export
# CSS classes for toggle state coloring
_TOGGLE_BG_OFF = str(bg_dui.error)    # Red when auto-play disabled
_TOGGLE_BG_ON = str(bg_dui.success)   # Green when auto-play enabled

def _toggle_color_js(toggle_id:str) -> str:  # JS snippet to sync toggle color classes
    """Generate JS to swap bg-error/bg-success on the toggle based on checked state."""
    return (
        f"var _t=document.getElementById('{toggle_id}');"
        f"if(_t){{_t.classList.remove('{_TOGGLE_BG_OFF}','{_TOGGLE_BG_ON}');"
        f"_t.classList.add(_t.checked?'{_TOGGLE_BG_ON}':'{_TOGGLE_BG_OFF}');}}"
    )

def render_auto_navigate_toggle(
    enabled:bool=False,  # Whether auto-navigate is enabled
) -> Any:  # Auto-navigate toggle component
    """Render auto-navigate toggle switch (client-side only, no server persistence)."""
    # Client-side JS to update auto-navigate flag + sync colors
    toggle_id = AudioControlIds.AUTO_NAV_TOGGLE
    color_js = _toggle_color_js(toggle_id)
    onchange_js = f"if(window.setReviewAutoNavigate) window.setReviewAutoNavigate(this.checked);{color_js}"

    # Only pass checked when enabled
    check_attr = {"checked": True} if enabled else {}
    color_cls = _TOGGLE_BG_ON if enabled else _TOGGLE_BG_OFF
    
    return Div(
        Label(
            Span(
                "Auto:",
                cls=combine_classes(font_size.sm, text_dui.base_content.opacity(70), m.r(2))
            ),
            Input(
                type="checkbox",
                id=toggle_id,
                name="auto_navigate",
                cls=combine_classes(toggle, toggle_sizes.sm, color_cls),
                onchange=onchange_js,
                **check_attr,
            ),
            cls=combine_classes(flex_display, items.center)
        ),
        cls=combine_classes(flex_display, items.center)
    )

## Combined Audio Controls

Renders both speed selector and auto-navigate toggle in a single component.

In [ ]:
#| export
def render_audio_controls(
    current_speed:float=1.0,  # Current playback speed
    auto_navigate:bool=False,  # Whether auto-navigate is enabled
    speed_url:str="",  # URL for speed changes
    oob:bool=False,  # Whether to render as OOB swap
) -> Any:  # Combined audio controls component
    """Render combined audio controls (speed selector + auto-navigate toggle)."""
    return Div(
        render_speed_selector(_REVIEW_SPEED_CONFIG, current_speed, speed_url),
        render_auto_navigate_toggle(auto_navigate),
        id=AudioControlIds.AUDIO_CONTROLS,
        cls=combine_classes(flex_display, items.center, gap(4)),
        hx_swap_oob="true" if oob else None,
    )

In [ ]:
from fasthtml.common import to_xml
import re

# Container delegates speed selector to cjm-fasthtml-web-audio's render_speed_selector.
# Verify the container wires the library correctly at the integration boundary:
# namespace-derived IDs/functions present, persisted speed flows through, OOB attr applied.

html_default = to_xml(render_audio_controls(current_speed=1.0, auto_navigate=False))
assert 'sd-review-speed-select' in html_default   # Library-derived ID
assert 'setReviewSpeed' in html_default            # Library-derived onchange JS
assert 'sd-review-auto-nav-toggle' in html_default
assert 'sd-review-audio-controls' in html_default
assert 'hx-post' not in html_default               # speed_url empty → no HTMX
assert 'setReviewSpeed(1.0)' not in html_default   # speed==1.0 → sync script is no-op

html_wired = to_xml(render_audio_controls(current_speed=2.0, auto_navigate=True, speed_url="/x/speed"))
assert 'hx-post="/x/speed"' in html_wired
assert 'hx-trigger="change"' in html_wired
assert 'setReviewSpeed(2.0)' in html_wired  # Initial sync script
assert re.search(r'<option[^>]*value="2.0"[^>]*selected', html_wired)

html_oob = to_xml(render_audio_controls(auto_navigate=True, oob=True))
assert 'hx-swap-oob' in html_oob
assert 'sd-review-audio-controls' in html_oob

print('Review audio controls tests passed')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()